# Systematics: GENIE

GENIE systematics are handled differently for uncertainty on the event rates vs. uncertainty on the xsec measurement

For the latter, we only consider the impact on response for the signal channel

In [ ]:
%load_ext autoreload
%autoreload 2

#print all output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
from os import path, makedirs
import sys
import uproot
from tqdm import tqdm
import datetime

# local imports
sys.path.append('../../')
from analysis_village.numucc_1p0pi.selection_definitions import *
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.makedf.util import *
from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from pyanalib.split_df_helpers import *
from pyanalib.stat_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.variable_calculator import get_cc1p0pi_tki
from pyanalib.pandas_helpers import pad_column_name
from makedf.constants import *
from makedf.geniesyst import *

plt.style.use("presentation.mplstyle")
cmap = mpl.cm.viridis
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredText
from matplotlib.offsetbox import AnchoredOffsetbox, DrawingArea, HPacker, VPacker, TextArea
from matplotlib.legend import Legend

# turn off PerformanceWarning
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_fig = False
today_str = datetime.datetime.now().strftime("%Y%m%d")
save_fig_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-genie-{}".format(today_str)
if not path.exists(save_fig_dir):
    makedirs(save_fig_dir)

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"

## -- MC 
mc_file = path.join(file_dir, "MC", "BNB_cosmics", "genie_wgts-CCQE", "aa_geniewgts_CCQE.df")
mc_split_df = pd.read_hdf(mc_file, key="split")
mc_n_split = get_n_split(mc_file)
print("mc_n_split: %d" %(mc_n_split))
print_keys(mc_file)

n_max_concat = 3
mc_keys2load = ['hdr', "mcnu", 'evt'] 
mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
mc_hdr_df = mc_dfs['hdr']
mc_nu_df = mc_dfs['mcnu']
mc_evt_df = mc_dfs['evt']

In [ ]:
## total pot
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))

# target_pot = 1e20
target_pot = mc_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))

mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))
mc_nu_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_nu_df))

In [ ]:
# add multiindex column index "mc" so that branch names match evt_df
# TODO: fix so I don't have to do this
mc_nu_df.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in mc_nu_df.columns])     # match # of column levels

In [ ]:
print("==== selected events ====")
mc_evt_df.loc[:,'nuint_categ'] = get_int_category(mc_evt_df)
mc_evt_df.loc[:,'genie_categ'] = get_genie_category(mc_evt_df)
print(mc_evt_df.nuint_categ.value_counts())
print(mc_evt_df.genie_categ.value_counts())

print("==== all events ====")
mc_nu_df.loc[:,'nuint_categ'] = get_int_category(mc_nu_df)
mc_nu_df.loc[:,'genie_categ'] = get_genie_category(mc_nu_df)
print(mc_nu_df.nuint_categ.value_counts())
print(mc_nu_df.genie_categ.value_counts())


In [ ]:
# regen_systematics

In [ ]:
syst_name = "GENIEReWeight_SBN_v1_multisim_RPA_CCQE"
mc_evt_df.mc[syst_name]

In [ ]:
def get_clipped_evts(df, var_col, bins):
    eps = 1e-8
    var = df[var_col]
    var = np.clip(var, bins[0], bins[-1] - eps)

    if 'pot_weight' in df.columns:
        weights = df.loc[:, 'pot_weight']
    else:
        print("No pot_weight column found, return 1 as pot scale")
        weights = np.ones_like(var)
    return var, weights


In [ ]:
var_config = VariableConfig.muon_momentum()

# Total MC reco muon momentum: for fake data
var_total_mc, weights_total_mc = get_clipped_evts(mc_evt_df, var_config.var_evt_reco_col, var_config.bins)

# --- all events, selected ---
# mc_evt_df divided into topology modes for subtraction from data in future
# first item in list is the signal topology
mc_evt_df_divided = [mc_evt_df[mc_evt_df.nuint_categ == mode]for mode in topology_list]

# Reco variable distribution for each 'nuint_categ' for stack plot and subtraction from the fake data
var_per_nuint_categ_mc, weights_per_categ = [], []
for mode in topology_list:
    var, weights = get_clipped_evts(mc_evt_df[mc_evt_df.nuint_categ == mode], var_config.var_evt_reco_col, var_config.bins)
    var_per_nuint_categ_mc.append(var)
    weights_per_categ.append(weights)

# Reco variable distribution for each genie mode
var_per_genie_mode_mc, weights_per_genie_mode = [], []
for mode in genie_mode_list:
    var, weights = get_clipped_evts(mc_evt_df[mc_evt_df.genie_categ == mode], var_config.var_evt_reco_col, var_config.bins)
    var_per_genie_mode_mc.append(var)
    weights_per_genie_mode.append(weights)

# --- signal events ---
# selected, for response matrix
# Signal event's reco muon momentum after the event selection
var_signal_sel_reco, weight_signal = get_clipped_evts(mc_evt_df[mc_evt_df.nuint_categ == 1], var_config.var_evt_reco_col, var_config.bins)

# Signal event's true muon momentum after the event selection
var_signal_sel_truth, weight_true_signal = get_clipped_evts(mc_evt_df[mc_evt_df.nuint_categ == 1], var_config.var_evt_truth_col, var_config.bins)

# total generated, for efficiency vector
# Signal event's true muon momentum without event selection
var_truth_signal, weight_truth_signal = get_clipped_evts(mc_nu_df[mc_nu_df.nuint_categ == 1], var_config.var_nu_col, var_config.bins)

In [ ]:
nevts_signal_truth, _, _ = plt.hist(var_truth_signal, bins=var_config.bins, weights=weight_truth_signal, histtype="step", label="True Signal")
nevts_signal_sel_reco, _, _ = plt.hist(var_signal_sel_reco, bins=var_config.bins, weights=weight_signal, histtype="step", label="Reco Selected Signal", color="k")
nevts_signal_sel_truth, _, _ = plt.hist(var_signal_sel_truth, bins=var_config.bins, weights=weight_signal, histtype="step", label="True Selected Signal")
plt.legend()
plt.ylabel("Events / Bin")
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
if save_fig:
    plt.savefig("{}/{}-sel_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show();

In [ ]:
# TODO: load xsec unit factor as a global variable, instead of calculating it every time
def get_genie_univs(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name):
    # cov_type = "xsec"
    XSEC_UNIT = 1e-38

    if cov_type == "xsec":
        print("generating covariance for xsec, using scale factor: {}".format(XSEC_UNIT))
        scale_factor = XSEC_UNIT

    elif cov_type == "rate":
        scale_factor = 1.0

    else:
        raise ValueError("Invalid covariance type: {}, choose xsec or rate".format(cov_type))

    n_univ = 100


    signal_cv = nevts_signal_sel_reco * scale_factor # = Response @ true_signal

    Covariance_Frac = np.zeros((len(signal_cv), len(signal_cv)))
    Covariance = np.zeros((len(signal_cv), len(signal_cv)))

    univ_events = []
    univ_effs = []
    univ_smears = []
    for uidx in range(n_univ):
        # univ_col_evt = ("mc", syst_name, "univ_{}".format(uidx)) #, "", "", "", "") #, "", "")
        # univ_col_mc = ("mc", syst_name, "univ_{}".format(uidx), "")
        univ_col = ("mc", syst_name, "univ_{}".format(uidx))

        # ---- uncertainty on the signal rate ----
        # GENIE syst need special treatment
        # we don't want uncertainty on the xsec
        # only consider its effect on the response matrix
        if cov_type == "xsec":
            true_signal_univ, _ = np.histogram(var_truth_signal, bins=var_config.bins, 
                                            weights=weight_truth_signal*mc_nu_df[mc_nu_df.nuint_categ == 1][univ_col])
            
            # new response matrix for univ
            bins2d = [var_config.bins, var_config.bins]
            reco_vs_true = get_smear_matrix(var_signal_sel_truth, var_signal_sel_reco, bins2d,
                                            weights=mc_evt_df[mc_evt_df.nuint_categ == 1][univ_col], plot=False)
            univ_smears.append(reco_vs_true)

            eff = get_eff(reco_vs_true, true_signal_univ) 
            univ_effs.append(eff)

            Response_univ = get_response_matrix(reco_vs_true, eff, var_config.bins, plot=False)
            signal_univ = Response_univ @ nevts_signal_truth # note that we multiply the CV signal rate!
            # signal_univ = signal_cv

        # for other systs, we just take the univ signal event rate
        elif cov_type == "rate":
            signal_univ, _ = np.histogram(var_signal_sel_reco, bins=var_config.bins, 
                                                weights=mc_evt_df[mc_evt_df.nuint_categ == 1][univ_col])

        else:
            raise ValueError("Invalid covariance type: {}, choose xsec or rate".format(cov_type))

        signal_univ = np.array(signal_univ) * scale_factor

        # ---- uncertainty on the background rate ----
        # loop over background categories
        # + univ background - cv background
        # note: cv background subtraction cancels out with the cv background subtraction for the cv event rate. 
        #       doing it anyways for the plot of universes on background subtracted event rate.
        for this_mc_evt_df in mc_evt_df_divided[1:]:
            weights = this_mc_evt_df[univ_col].copy()
            weights[np.isnan(weights)] = 1 ## IMPORTANT: make nan weights to 1. to ignore them
            this_var = this_mc_evt_df[var_config.var_evt_reco_col]
            this_var = np.clip(this_var, var_config.bins[0], var_config.bins[-1] - eps)
            background_univ, _ = np.histogram(this_var, bins=var_config.bins, weights=weights)
            background_cv, _ = np.histogram(this_var, bins=var_config.bins)
            background_univ = np.array(background_univ) * scale_factor
            background_cv = np.array(background_cv) * scale_factor
            signal_univ += background_univ - background_cv

        univ_events.append(signal_univ)

    univ_events = np.array(univ_events)
    return univ_events, signal_cv

In [ ]:
# Note: these are background subtracted
univ_events, cv_events = get_genie_univs("xsec", mc_evt_df, mc_nu_df, var_config, syst_name)
ret_xsec = get_covariance(univ_events, cv_events)
plot_univ_hist(mc_evt_df, var_config, ("mc", syst_name), univ_events, cv_events)

univ_events, cv_events = get_genie_univs("rate", mc_evt_df, mc_nu_df, var_config, syst_name)
ret_rate = get_covariance(univ_events, cv_events)
plot_univ_hist(mc_evt_df, var_config, ("mc", syst_name), univ_events, cv_events)

frac_unc = np.sqrt(np.diag(ret_xsec["Covariance_Frac"]))
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_unc, histtype="step", color="black")
frac_unc = np.sqrt(np.diag(ret_rate["Covariance_Frac"]))
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_unc, histtype="step", color="gray")

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Fractional Uncertainty")
plt.grid(True)
plt.show();

In [ ]:
matrix_type = "Covariance"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret_xsec[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)
plot_heatmap(ret_rate[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)